<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Refactoring Spaghetti Code into Clean, Modular Code

## Introduction
When begin coding, it's completely normal to create one long, linear script that “gets the job done.”

But over time, this results in:

- spaghetti code (tangled logic)
- repeated hardcoded values
- difficulty reusing code
- challenging debugging
- no clear structure

Refactoring is the process of restructuring code without changing its behaviour, improving readability, modularity, and maintainability.

## Steps to refactoring: 

### Understand what the current code actually does
Before changing anything:

- Identify the main tasks the script performs.
- Note dependencies, inputs, outputs, and side effects.
- Write down expected behaviour (even informally).
- Add comments to clarify intent if needed.

This forms your map for refactoring.


### Add tests (even basic ones)
You don't need full CI/CD yet — just:

- Smoke tests: “Does the code run end‑to‑end without errors?”
- Golden tests: fixed inputs → fixed expected outputs
- Tests for critical functions or edge cases

Why?

Tests give you confidence to refactor without breaking stuff.

### Improve readability inside the existing script
Without changing the architecture yet:

- Remove dead code
- Improve variable names
- Introduce varaiables instead of hardcoding values
- Format according to PEP8 (Python) or relevant style guide

This makes the next steps a lot easier.

### Extract functions (the key turning point)
Start grouping logic logically:

- One function = one responsibility
- Each function has clear inputs and outputs


Example:
```python
def load_data(path): ...
def clean_data(df): ...
def transform_data(df): ...
def write_output(df, dest): ...
```


***This is the foundation of modular coding.***


### Introduce modular structure
This is where you break the big script into multiple files:

- `utils.py` for shared helpers
- `data_loading.py`
- `data_processing.py`
- `main.py` to orchestrate everything

### Introduce configuration management
Replace hardcoded values with:

- config files (YAML, JSON, TOML)
- environment variables
- parameterised scripts

**This enables scalability and reusability.**

### Introduce logging instead of print()
Logging gives:

- levels (`info`, `debug`, `error`)
- `timestamps`
- ability to save logs to files

**This improves debuggability and production readiness.**

### Introduce error handling
Add:

- `try/except` blocks where appropriate
- custom error messages
- recovery logic if needed

This makes your code resilient.

### Package and structure the project
Depending on the use-case:

- Create a Python package
- Use `__init__.py`
- Add a proper folder layout (src/ structure)

**This is the step before “scripts calling each other” becomes natural and clean.**

### (Optional but recommended) Add automation

- Add a Makefile or task runner
- Add tests + linting into CI/CD pipeline
- Include documentation (README, docstrings, API docs)


**In short, the stages look like:**

- Understand current code
- Add tests
- Clean readability
- Extract functions
- Modularise (yes!)
- Add configuration
- Add logging
- Add error handling
- Package the project
- Add automation & documentation

**Example:**

In [ ]:
import pandas as pd
df = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/train_titanic.csv")
avg = df["Fare"].mean()
df["AboveAverageFare"] = df["Fare"] > avg
df = df[df["AboveAverageFare"] == True]
df.to_csv("titanic_high_fare.csv", index=False)
print("done")

Issues:

- all logic mixed together
- hardcoded file names
- no functions
- no structure for reuse
- not testable

Lets analyse the code: 

- Understand what the current code actually does
- Add tests (even basic ones)
- Improve readability inside the existing script


In [ ]:
##Removing hardcoded values: 
import pandas as pd

INPUT_PATH = "https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/train_titanic.csv"
OUTPUT_PATH = "titanic_high_fare.csv"

df = pd.read_csv(INPUT_PATH)
avg = df["Fare"].mean()
df["AboveAverageFare"] = df["Fare"] > avg
df = df[df["AboveAverageFare"] == True]
df.to_csv(OUTPUT_PATH, index=False)
print("done")

- Extract functions (the key turning point)


In [ ]:
import pandas as pd

INPUT_PATH = "https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/train_titanic.csv"
OUTPUT_PATH = "titanic_high_fare.csv"

def load_data(path):
    df = pd.read_csv(path)
    return df

def compute_above_average_fare(df):
    avg = df["Fare"].mean()
    df["AboveAverageFare"] = df["Fare"] > avg
    return df

def filter_high_fare(df):
    df = df[df["AboveAverageFare"] == True]
    return df

def save_data(df, path):
    df.to_csv(path, index=False)

def main():
    ## Initialising constants
    input_path = INPUT_PATH
    output_path = OUTPUT_PATH
    ## Running the pipeline
    df = load_data(input_path)
    df = compute_above_average_fare(df)
    df = filter_high_fare(df)
    save_data(df, output_path)

    print("Done.")

if __name__ == "__main__":
    main()

## Refactoring

Now lets use the following project structure as a reference: 


```python
titanic_project/
    main.py
    src/
        __init__.py
        config.py
        io_handler.py
        preprocessing.py
        pipeline.py
```


- `config.py`: Central definition of parameters and paths. This script stores configuration variables and constants used across the project. It includes definitions for column names, test sizes, random states, and file names for models and logs, and paths to datasets.

- `io_handler.py`: Reading, writing and basic cleaning of raw data. Initial splitting(e.g., sampling on large datasets). This script is responsible for extracting the raw  dataset and load the processed dataset. It contains functions to read and write the data from/to a specified file path.

- `preprocessing.py`: This script contains functions for data cleaning and preprocessing. It handles tasks such as converting data types, encoding categorical variables, scaling numerical features, and splitting the data into training and testing sets.

- `pipeline.py`: Orchestrating pipeline stages in sequence. This script defines and orchestrates the end to end pipeline for the project. It encapsulates the sequential steps from data loading and preprocessing.

- `main.py`: Entry point for triggering the pipeline.This is the main entry point for the project. It orchestrates the entire process by calling the main pipeline function and handling overall execution flow.


Lets group our code to follow the structure previously mentioned. 

In [ ]:
import pandas as pd

# ==========================================
# SECTION: CONFIG (The source for src/config.py)
# ==========================================
INPUT_PATH = "https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/train_titanic.csv"
OUTPUT_PATH = "titanic_high_fare.csv"

# ==========================================
# SECTION: IO_HANDLER (The source for src/io_handler.py)
# ==========================================
def load_data(path):
    df = pd.read_csv(path)
    return df

def save_data(df, path):
    df.to_csv(path, index=False)
    
# ==========================================
# SECTION: PREPROCESSING (The source for src/preprocessing.py)
# ==========================================

def compute_above_average_fare(df):
    avg = df["Fare"].mean()
    df["AboveAverageFare"] = df["Fare"] > avg
    return df

def filter_high_fare(df):
    df = df[df["AboveAverageFare"] == True]
    return df

def clean_and_enrich_churn_data(df):
    df_clean = compute_above_average_fare(df)
    df_clean = filter_high_fare(df_clean)
    return df_clean

# ==========================================
# SECTION: PIPELINE (The source for src/pipeline.py)
# ==========================================
def run_de_pipeline(input_path, output_path):
    raw_df = load_data(input_path)
    processed_df = clean_and_enrich_churn_data(raw_df)
    save_data(processed_df, output_path)

# ==========================================
# SECTION: MAIN (The source for main.py)
# ==========================================

def main():
    input_path = INPUT_PATH
    output_path = OUTPUT_PATH
    run_de_pipeline(input_path, output_path)
    print("Done.")

if __name__ == "__main__":
    main()

## Exercise: From Monolith to Modular Pipeline

### Background

You have been given a "Spaghetti" script designed to process customer churn data. Currently, this script performs several critical Data Engineering tasks in a single, linear file:

* It initializes a distributed computing environment.
* It extracts raw data from a remote cloud URL.
* It cleans the data (type casting and missing value imputation).
* It performs feature engineering and target standardization.
* It exports the final "Gold" dataset to a local directory.

While the script "works," it is difficult to maintain, impossible to unit test, and hardcoded to a specific dataset.

### Your Mission

Your goal is to refactor this monolith into a **Modular Project Structure**. You will evolve the code through a functional cleanup phase and finally "explode" it into a professional directory structure that separates **Configuration**, **Logic**, and **Orchestration**.

---

### Phase 1: The Functional Cleanup

Before splitting the files, you must group the "Spaghetti" logic into named functions. This helps identify the **Single Responsibility** of each block of code.

**Task:** Group your logic into the following sections within a single script:

1. **Configuration Section:** Define all paths and column lists as constants.
2. **IO Handler Section:** Create functions for `extract_from_csv` and `load_to_csv`.
3. **Preprocessing Section:** Create functions for `handle_types`, `impute_values`, and `add_features`.
4. **Pipeline Section:** Create a master function `run_de_pipeline` that calls the others in order.

---

### Phase 2: The Project Refactor

Now that your logic is modular, you will move these sections into a professional folder structure. This decoupling allows different teams to work on different parts of the pipeline simultaneously.

**Task:** Create the following project structure and distribute your code:

* **`src/config.py`**: Move all hardcoded variables here.
* **`src/io_handler.py`**: Move all data reading and writing functions here.
* **`src/preprocessing.py`**: Move all data cleaning and transformation logic here.
* **`src/pipeline.py`**: Move the orchestration function here.
* **`main.py`**: This should be your only entry point. It should import the `SparkSession`, import the `config` and `pipeline` modules, and trigger the execution.

---

### Success Criteria

1. **Zero Hardcoding:** No URLs or Column names should exist in `preprocessing.py` or `main.py` (they must come from `config.py`).
2. **Clean Execution:** Running `python main.py` should trigger the full Spark pipeline and produce a success log.
3. **Traceability (Optional):** Your terminal should show clear `INFO` logs for each stage of the process (Extraction  Preprocessing  Saving).

### Discussion Questions for the End

* If we decided to change our imputation strategy from *Mean* to *Median*, which file would we modify?
* If the S3 URL for the raw data changes, how many files do we need to edit?
* Why do we keep the `SparkSession` initialization in `main.py` instead of inside the `preprocessing` functions?

In [ ]:
##Original code
import pandas as pd
import numpy as np

# Using pandas to read the CSV directly
df = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/WA_Fn-UseC_-Telco-Customer-Churn.csv")
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    mean_val = df[col].mean()
    df[col] = df[col].fillna(mean_val)
cat_list = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService']
for col in cat_list:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
df['MonthlyChargeRatio'] = df['TotalCharges'] / (df['tenure'] + 1)
df['churn_binary'] = np.where(df['Churn'] == 'Yes', 1, np.where(df['Churn'] == 'No', 0, np.nan))
final_cols =  ['tenure', 'MonthlyCharges', 'TotalCharges'] + cat_list + ['MonthlyChargeRatio', 'churn_binary']
#final_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'gender', 'MonthlyChargeRatio', 'churn_binary']
final_pdf = df[final_cols]
final_pdf.to_csv("churn_processed.csv", index=False)

In [ ]:
##Solution phase 1:
import pandas as pd
import logging
import os
from pathlib import Path


# Setup logging exactly like your scripts
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==========================================
# SECTION: CONFIG (The source for config.py)
# ==========================================
TARGET: str = 'Churn'

NUMERIC_COLS: list[str] = ['tenure', 'MonthlyCharges', 'TotalCharges']

CATEGORICAL_COLS: list[str] = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 
    'PhoneService', 'MultipleLines', 'InternetService'
]

# The final schema for the "Gold" table (Analytics/Modeling ready)
COLUMNS_TO_KEEP: list[str] = NUMERIC_COLS + CATEGORICAL_COLS + ['MonthlyChargeRatio', 'churn_binary']
#Root path:
# This works when running as a script (e.g., python main.py)
#Assumes script is nested 2 levels deep (e.g. /src/pipeline/script.py)
#project_root = Path(__file__).resolve().parent.parent

# This works in Jupyter Notebooks
project_root = Path.cwd()
# For pytthon scipts:
#project_root = Path(__file__).resolve().parent.parent

# File Paths
##Data sources
RAW_DATA_PATH = "https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/WA_Fn-UseC_-Telco-Customer-Churn.csv"
##Pre-process data path
CLEAN_DATA_PATH: str = "data/processed/churn_preprocessed.csv"
PROCESSED_DATA_PATH = project_root / CLEAN_DATA_PATH

# ==========================================
# SECTION: IO_HANDLER (The source for io_handler.py)
# ==========================================
def extract_from_csv(filepath: str) -> pd.DataFrame:
    """
    Extracts data from a CSV file located locally or via a public URL.

    Args:
        filepath (str): The local path or public URL to the raw CSV file.

    Returns:
        pd.DataFrame: A pandas DataFrame containing the raw dataset.
    """
    logger.info(f"Initiating extraction from: {filepath}")
    
    if not filepath.startswith(('http://', 'https://')):
        if not os.path.exists(filepath):
            logger.error(f"File not found: {filepath}")
            raise FileNotFoundError(f"Could not find local file at {filepath}")
    try:
        df = pd.read_csv(filepath)
        logger.info(f"Extraction successful. Loaded {len(df)} rows.")
        return df
    except Exception as e:
        logger.error(f"Failed to load data from {filepath}: {e}")
        raise RuntimeError(f"Data extraction failed: {e}")


def load_to_csv(df: pd.DataFrame, filepath: str) -> None:
    """
    Saves a DataFrame to a local CSV file. Automatically creates directories if missing.

    Args:
        df (pd.DataFrame): The processed DataFrame to save.
        filepath (str): The local path where the file should be saved.
    """
    try:
        # Create the folder if it doesn't exist (e.g., data/processed/)
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        
        df.to_csv(filepath, index=False)
        logger.info(f"Successfully loaded data to: {filepath}")
    except Exception as e:
        logger.error(f"Failed to save data to {filepath}: {e}")
        raise RuntimeError(f"Data loading failed: {e}")
    
    
# ==========================================
# SECTION: PREPROCESSING (The source for preprocessing.py)
# ==========================================
def handle_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converts specific columns to the correct data types for analysis.

    Args:
        df (pd.DataFrame): The DataFrame to process.

    Returns:
        pd.DataFrame: DataFrame with corrected column types.
    """
    if 'TotalCharges' in df.columns:
        logger.info("Converting 'TotalCharges' to numeric.")
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    return df

def impute_values(df: pd.DataFrame, 
                  num_cols: list[str], 
                  cat_cols: list[str]) -> pd.DataFrame:
    """
    Fills missing values using mean for numeric and mode for categorical columns.

    Args:
        df (pd.DataFrame): The DataFrame containing null values.
        num_cols (list[str]): List of numeric columns to impute with mean.
        cat_cols (list[str]): List of categorical columns to impute with mode.

    Returns:
        pd.DataFrame: DataFrame with no remaining missing values in specified columns.
    """
    logger.info("Imputing missing values.")
    for col in [c for c in num_cols if c in df.columns]:
        df[col] = df[col].fillna(df[col].mean())
    
    for col in [c for c in cat_cols if c in df.columns]:
        if not df[col].mode().empty:
            df[col] = df[col].fillna(df[col].mode()[0])
    return df

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates new business metrics from existing raw data.

    Args:
        df (pd.DataFrame): The cleaned DataFrame.

    Returns:
        pd.DataFrame: DataFrame with the 'MonthlyChargeRatio' feature added.
    """
    logger.info("Engineering new features.")
    if 'TotalCharges' in df.columns and 'tenure' in df.columns:
        df['MonthlyChargeRatio'] = df['TotalCharges'] / (df['tenure'] + 1)
    return df

def format_target(df: pd.DataFrame, 
                  target: str) -> pd.DataFrame:
    """
    Maps the target string labels to binary integers.

    Args:
        df (pd.DataFrame): The DataFrame containing the target.
        target (str): The name of the target column (e.g., 'Churn').

    Returns:
        pd.DataFrame: DataFrame with a new 'churn_binary' column.
    """
    if target in df.columns:
        logger.info(f"Standardizing target column: {target}")
        df['churn_binary'] = df[target].map({'Yes': 1, 'No': 0})
    return df

def clean_and_enrich_churn_data(
    df: pd.DataFrame, 
    target: str, 
    numeric_cols: list[str], 
    categorical_cols: list[str],
    final_columns: list[str]
) -> pd.DataFrame:
    """
    Cleans data types, imputes missing values, and creates calculated fields.

    Args:
        df (pd.DataFrame): The raw input DataFrame.
        target (str): The name of the original target column.
        numeric_cols (list[str]): List of numeric column names.
        categorical_cols (list[str]): List of categorical column names.
        final_columns (list[str]): The final list of columns to return.

    Returns:
        pd.DataFrame: A cleaned and enriched DataFrame with the specified schema.
    """
    logger.info(f"Starting preprocessing. Input shape: {df.shape}")
    
    # Start with a copy to avoid changing the original data
    df_clean = df.copy()
    
    # Execute steps one by one
    df_clean = handle_types(df_clean)
    df_clean = impute_values(df_clean, numeric_cols, categorical_cols)
    df_clean = add_features(df_clean)
    df_clean = format_target(df_clean, target)
    
    # Select final columns
    result_df = df_clean[final_columns]
    
    logger.info(f"Preprocessing complete. Final shape: {result_df.shape}")
    return result_df

# ==========================================
# SECTION: PIPELINE (The source for pipeline.py)
# ==========================================
def run_de_pipeline(
    input_path: str, 
    output_path: str, 
    target: str, 
    num_cols: list[str], 
    cat_cols: list[str], 
    final_cols: list[str]
) -> None:
    """
    Runs the end-to-end Data Engineering pipeline.

    Args:
        input_path (str): Path or URL to the raw data.
        output_path (str): Path where the processed CSV will be saved.
        target (str): Name of the target column.
        num_cols (list[str]): List of numeric feature names.
        cat_cols (list[str]): List of categorical feature names.
        final_cols (list[str]): Final list of columns to export.
    """
    logger.info("Starting Data Engineering Pipeline...")
    
    # 1. Extraction
    raw_df = extract_from_csv(input_path)
    
    # 2. Transformation (Cleaning, Enrichment, and Selection)
    processed_df = clean_and_enrich_churn_data(
        df=raw_df,
        target=target,
        numeric_cols=num_cols,
        categorical_cols=cat_cols,
        final_columns=final_cols
    )
    
     # 3. Loading
    load_to_csv(processed_df, output_path)
    #processed_df.to_csv(output_path, index=False)
    logger.info(f"Pipeline finished. Processed data saved to: {output_path}")

# ==========================================
# SECTION: MAIN (The source for main.py)
# ==========================================
def main() -> None:
    """
    Orchestrates the pipeline execution by passing configurations to the pipeline function.
    """
    try:
        run_de_pipeline(
            input_path=RAW_DATA_PATH,
            output_path=PROCESSED_DATA_PATH,
            target=TARGET,
            num_cols=NUMERIC_COLS,
            cat_cols=CATEGORICAL_COLS,
            final_cols=COLUMNS_TO_KEEP
        )
        print("Success: Churn pipeline completed!")
        logger.info("Success: Churn pipeline completed!")
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        #logging.error(f"Pipeline failed: {e}")

if __name__ == "__main__":
    main()

2026-02-17 09:11:50,647 - INFO - Starting Data Engineering Pipeline...
2026-02-17 09:11:50,650 - INFO - Initiating extraction from: https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/WA_Fn-UseC_-Telco-Customer-Churn.csv
2026-02-17 09:11:50,952 - INFO - Extraction successful. Loaded 7043 rows.
2026-02-17 09:11:50,953 - INFO - Starting preprocessing. Input shape: (7043, 21)
2026-02-17 09:11:50,957 - INFO - Converting 'TotalCharges' to numeric.
2026-02-17 09:11:50,963 - INFO - Imputing missing values.
2026-02-17 09:11:50,973 - INFO - Engineering new features.
2026-02-17 09:11:50,976 - INFO - Standardizing target column: Churn
2026-02-17 09:11:50,979 - INFO - Preprocessing complete. Final shape: (7043, 12)
2026-02-17 09:11:51,010 - INFO - Successfully loaded data to: c:\Users\MiguelAngelSanchezRa\Python_Sandbox\CBS_Python_eng\04_04_From_Notebooks_to_Production_Pipelines_2\data\processed\churn_preprocessed.csv
2026-02-17 09:11:51,013 - INFO - Pipeline finished. Processed

Success: Churn pipeline completed!


---

### Phase 2: The Project Refactor

Now that your logic is modular, you will move these sections into a professional folder structure. This decoupling allows different teams to work on different parts of the pipeline simultaneously.

**Task:** Create the following project structure and distribute your code:

* **`src/config.py`**: Move all hardcoded variables here.
* **`src/io_handler.py`**: Move all data reading and writing functions here.
* **`src/preprocessing.py`**: Move all data cleaning and transformation logic here.
* **`src/pipeline.py`**: Move the orchestration function here.
* **`main.py`**: This should be your only entry point. It should import the `SparkSession`, import the `config` and `pipeline` modules, and trigger the execution.



